# GPU 커널, Triton으로 직접 짜 보기

<a href="https://colab.research.google.com/github/osehyeon/AI-JBNU-2026/blob/main/notebooks/triton_lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

파이썬으로 GPU 커널을 쓰고, 왜 빨라지는지 직접 잽니다. 끝나면 내 GPU에서 잰 결과 그래프가 남습니다.

### 시작하기

1. 메뉴 **런타임 > 런타임 유형 변경**에서 **T4 GPU**를 고르고 저장합니다.
2. 위에서부터 셀 왼쪽의 ▶를 눌러 하나씩 실행합니다. 경고가 나오면 **계속 실행**을 누릅니다.
3. 수업을 놓쳤으면 메뉴 **런타임 > 모두 실행**을 누릅니다. `TODO`가 적힌 줄을 고치고 그 셀을 다시 실행해 보세요.

## 1. GPU 확인하기

GPU 이름이 나오면 성공입니다. 오류가 나오면 런타임 유형을 GPU로 바꾸고 다시 실행합니다.

In [ ]:
#@title 1. GPU 확인
import subprocess
import sys

try:
    import triton
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "triton"], check=True)
    import triton

import matplotlib.pyplot as plt
import torch
import triton.language as tl
import triton.testing

if not torch.cuda.is_available():
    raise RuntimeError("GPU가 연결되지 않았습니다. 런타임 > 런타임 유형 변경에서 T4 GPU를 고르세요.")

gpu_name = torch.cuda.get_device_name()
results = {}
print(f"GPU: {gpu_name}")
print(f"torch {torch.__version__}, triton {triton.__version__}")

## 2. 같은 Softmax인데 왜 시간이 다를까

Softmax는 점수를 확률로 바꾸는 함수입니다. 같은 계산을 세 가지로 씁니다. 결과는 같은데 시간과 **GPU에 띄운 커널 수**가 다릅니다.

- `naive`: PyTorch 연산을 하나씩 부릅니다
- `torch.softmax`: PyTorch에 이미 있는 함수입니다
- `compiled`: `naive`를 `torch.compile`로 컴파일합니다 (`dynamic=False`: 입력 크기마다 따로 컴파일)

처음에는 컴파일 때문에 몇 초 더 걸립니다.

In [ ]:
#@title 2. Softmax 세 가지 비교
def naive_softmax(x):
    x_max = x.max(dim=1, keepdim=True)[0]
    num = torch.exp(x - x_max)
    return num / num.sum(dim=1, keepdim=True)


def builtin_softmax(x):
    return torch.softmax(x, dim=1)


compiled_softmax = torch.compile(naive_softmax, dynamic=False)


def kernel_launches(fn):
    fn()
    torch.cuda.synchronize()
    activities = [torch.profiler.ProfilerActivity.CPU, torch.profiler.ProfilerActivity.CUDA]
    with torch.profiler.profile(activities=activities) as prof:
        fn()
        torch.cuda.synchronize()
    names = [e.name for e in prof.events()]
    launches = sum("LaunchKernel" in n for n in names)
    triton_names = sorted({n for n in names if n.startswith("triton_")})
    return launches, triton_names


x = torch.randn(4096, 16384, device="cuda")
assert torch.allclose(naive_softmax(x), builtin_softmax(x), atol=1e-5)
assert torch.allclose(naive_softmax(x), compiled_softmax(x), atol=1e-5)

print(f"{'':>14} {'ms':>9} {'kernels':>8}")
for name, fn in [("naive", naive_softmax), ("torch.softmax", builtin_softmax), ("compiled", compiled_softmax)]:
    ms = triton.testing.do_bench(lambda: fn(x))
    launches, triton_names = kernel_launches(lambda: fn(x))
    print(f"{name:>14} {ms:>9.3f} {launches:>8}")
    for n in triton_names:
        print(f"{'':>14} -> {n}")

`naive`는 연산 하나마다 커널을 하나씩 띄우고, 그때마다 GPU 메모리를 읽고 씁니다. `compiled`는 연산을 커널 하나로 묶어서 한 번 읽고 한 번 씁니다.

`triton_`으로 시작하는 이름이 보이나요? `torch.compile`이 만든 커널이 바로 **Triton 커널**입니다. 이제 이 커널을 직접 씁니다. 직접 쓴 커널을 **custom kernel**이라고 부릅니다.

## 3. 실습 1: 벡터 덧셈

세 단어만 보면 됩니다.

- `program_id`: 나는 몇 번째 프로그램인가
- `offsets`: 내가 맡은 원소들의 번호
- `mask`: 배열 끝을 넘는 자리는 읽지도 쓰지도 않는다

프로그램 하나가 원소 `BLOCK_SIZE`개를 맡습니다.

In [ ]:
#@title 3. 벡터 덧셈 커널
@triton.jit
def add_kernel(x_ptr, y_ptr, out_ptr, n_elements, BLOCK_SIZE: tl.constexpr):
    pid = tl.program_id(axis=0)
    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n_elements
    x = tl.load(x_ptr + offsets, mask=mask)
    y = tl.load(y_ptr + offsets, mask=mask)
    tl.store(out_ptr + offsets, x + y, mask=mask)


def add(x, y, block_size=1024):
    out = torch.empty_like(x)
    n = out.numel()
    grid = (triton.cdiv(n, block_size),)
    add_kernel[grid](x, y, out, n, BLOCK_SIZE=block_size)
    return out


x = torch.rand(98432, device="cuda")
y = torch.rand(98432, device="cuda")
print("정답과 같은가:", torch.allclose(add(x, y), x + y))

### 해 보기 1

`BLOCK_SIZE`를 바꿔 가며 잽니다. 프로그램 수는 원소 수를 `BLOCK_SIZE`로 나눠 올림한 값입니다.

In [ ]:
#@title 해 보기 1: BLOCK_SIZE 바꾸기
n = 10**6  # TODO: 10**8 로도 바꿔 보기
x = torch.rand(n, device="cuda")
y = torch.rand(n, device="cuda")

results["add"] = {}
print(f"{'BLOCK_SIZE':>10} {'programs':>9} {'ms':>9} {'GB/s':>7}")
for block_size in [64, 128, 256, 512, 1024, 2048]:
    ms = triton.testing.do_bench(lambda: add(x, y, block_size))
    gbps = 3 * x.numel() * x.element_size() / ms * 1e-6
    results["add"][block_size] = gbps
    print(f"{block_size:>10} {triton.cdiv(n, block_size):>9} {ms:>9.4f} {gbps:>7.0f}")

## 4. 실습 2: Softmax custom kernel 쓰기

덧셈과 달리 프로그램 하나가 **행 하나**를 맡습니다. 행 전체의 최댓값과 합이 필요하기 때문입니다.

아래 셀의 `TODO` **한 줄만** 고치면 됩니다. `num`을 행 전체의 합으로 나눕니다. 행 전체의 합은 `tl.sum(num, axis=0)`입니다.

In [ ]:
#@title 4. Softmax 커널 (TODO 한 줄)
@triton.jit
def softmax_kernel(out_ptr, in_ptr, in_stride, out_stride, n_cols, BLOCK_SIZE: tl.constexpr):
    row = tl.program_id(0)
    offs = tl.arange(0, BLOCK_SIZE)
    mask = offs < n_cols
    x = tl.load(in_ptr + row * in_stride + offs, mask=mask, other=-float("inf"))
    x = x - tl.max(x, axis=0)
    num = tl.exp(x)
    y = num  # TODO: 이 줄을 Softmax 식으로 바꾸기
    tl.store(out_ptr + row * out_stride + offs, y, mask=mask)


def run_softmax(kernel, x):
    n_rows, n_cols = x.shape
    y = torch.empty_like(x)
    block = triton.next_power_of_2(n_cols)
    num_warps = 4 if block < 2048 else (8 if block < 4096 else 16)
    kernel[(n_rows,)](y, x, x.stride(0), y.stride(0), n_cols, BLOCK_SIZE=block, num_warps=num_warps)
    return y


x = torch.randn(1823, 781, device="cuda")
correct = torch.allclose(run_softmax(softmax_kernel, x), builtin_softmax(x), atol=1e-6)
print("정답과 같은가:", correct)
if correct:
    custom_kernel = softmax_kernel
else:
    print("아직 다릅니다. TODO 줄을 고치고 이 셀을 다시 실행하세요.")

막히면 아래 셀의 `코드 표시`를 눌러 정답을 봅니다. 위 커널이 아직 맞지 않으면 그래프에는 이 정답 커널의 시간이 들어갑니다.

In [ ]:
#@title 정답 보기
@triton.jit
def softmax_answer(out_ptr, in_ptr, in_stride, out_stride, n_cols, BLOCK_SIZE: tl.constexpr):
    row = tl.program_id(0)
    offs = tl.arange(0, BLOCK_SIZE)
    mask = offs < n_cols
    x = tl.load(in_ptr + row * in_stride + offs, mask=mask, other=-float("inf"))
    x = x - tl.max(x, axis=0)
    num = tl.exp(x)
    y = num / tl.sum(num, axis=0)
    tl.store(out_ptr + row * out_stride + offs, y, mask=mask)


x = torch.randn(1823, 781, device="cuda")
print("정답 커널이 torch.softmax와 같은가:", torch.allclose(run_softmax(softmax_answer, x), builtin_softmax(x), atol=1e-6))
if not correct:
    custom_kernel = softmax_answer

### 해 보기 2

행 길이를 바꿔 가며 네 가지를 비교합니다. `naive`와 custom kernel은 몇 배 차이가 나나요? `compiled`, `torch.softmax`와는 어떤가요?

In [ ]:
#@title 해 보기 2: 행 길이 바꾸기
methods = {
    "naive": naive_softmax,
    "torch.softmax": builtin_softmax,
    "compiled": compiled_softmax,
    "custom kernel": lambda x: run_softmax(custom_kernel, x),
}
results["softmax"] = {name: {} for name in methods}
print(f"{'n_cols':>7}" + "".join(f"{name:>15}" for name in methods) + "  (ms)")
for n_cols in [256, 1024, 4096, 16384]:  # TODO: 32768 도 넣어 보기
    x = torch.randn(4096, n_cols, device="cuda")
    try:
        times = {name: triton.testing.do_bench(lambda: fn(x)) for name, fn in methods.items()}
    except Exception as e:
        print(f"{n_cols:>7} 실패 ({type(e).__name__}): 이 GPU에는 블록이 너무 큽니다")
        continue
    for name, ms in times.items():
        results["softmax"][name][n_cols] = ms
    print(f"{n_cols:>7}" + "".join(f"{ms:>15.3f}" for ms in times.values()))

## 5. 내 GPU 결과 그래프

지금까지 잰 값을 한 장으로 그립니다. 이 그림이 오늘의 결과물입니다. 그림을 우클릭해 저장할 수 있습니다.

In [ ]:
#@title 5. 결과 그래프
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))

add_results = results["add"]
axes[0].bar([str(k) for k in add_results], list(add_results.values()), color="#F26B43")
axes[0].set_xlabel("BLOCK_SIZE")
axes[0].set_ylabel("GB/s")
axes[0].set_title("Vector add")

colors = {"naive": "#6B7280", "torch.softmax": "#2E9E6B", "compiled": "#4A90C8", "custom kernel": "#F26B43"}
for name, color in colors.items():
    row = results["softmax"][name]
    axes[1].plot(list(row), list(row.values()), marker="o", color=color, label=name)
axes[1].set_xscale("log", base=2)
axes[1].set_yscale("log")
axes[1].set_xticks(list(results["softmax"]["naive"]), [str(k) for k in results["softmax"]["naive"]])
axes[1].minorticks_off()
axes[1].set_xlabel("n_cols")
axes[1].set_ylabel("time (ms)")
axes[1].set_title("Softmax, 4096 rows")
axes[1].legend(frameon=False)

for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
fig.suptitle(gpu_name)
fig.tight_layout()
fig.savefig("my_gpu_results.png", dpi=150)
plt.show()

## 6. 선택 과제: 행렬곱과 autotune

수업에서 다루지 않습니다. 더 해 보고 싶으면 아래 `RUN_MATMUL`을 `True`로 바꿔 실행합니다. 몇 분 걸릴 수 있습니다.

프로그램 `(pid_m, pid_n)`이 C의 `BM x BN` 타일 하나를 맡고, K 방향으로 `BK`씩 읽어 곱합니다. GPU에 비해 블록이 너무 크면 **실패**로 나옵니다.

In [ ]:
#@title 6. 행렬곱 커널과 autotune
RUN_MATMUL = False  # TODO: True 로 바꾸면 실행


@triton.jit
def matmul_kernel(a_ptr, b_ptr, c_ptr, M, N, K,
                  sam, sak, sbk, sbn, scm, scn,
                  BM: tl.constexpr, BN: tl.constexpr, BK: tl.constexpr):
    pid_m, pid_n = tl.program_id(0), tl.program_id(1)
    rm = pid_m * BM + tl.arange(0, BM)
    rn = pid_n * BN + tl.arange(0, BN)
    rk = tl.arange(0, BK)
    a_ptrs = a_ptr + rm[:, None] * sam + rk[None, :] * sak
    b_ptrs = b_ptr + rk[:, None] * sbk + rn[None, :] * sbn
    acc = tl.zeros((BM, BN), dtype=tl.float32)
    for k in range(0, tl.cdiv(K, BK)):
        a = tl.load(a_ptrs, mask=(rm[:, None] < M) & (rk[None, :] < K - k * BK), other=0.)
        b = tl.load(b_ptrs, mask=(rk[:, None] < K - k * BK) & (rn[None, :] < N), other=0.)
        acc += tl.dot(a, b)
        a_ptrs += BK * sak
        b_ptrs += BK * sbk
    c_ptrs = c_ptr + rm[:, None] * scm + rn[None, :] * scn
    tl.store(c_ptrs, acc.to(tl.float16), mask=(rm[:, None] < M) & (rn[None, :] < N))


def matmul(a, b, BM=64, BN=64, BK=32, num_warps=4, num_stages=3):
    M, K = a.shape
    _, N = b.shape
    c = torch.empty((M, N), device=a.device, dtype=torch.float16)
    grid = (triton.cdiv(M, BM), triton.cdiv(N, BN))
    matmul_kernel[grid](a, b, c, M, N, K,
                        a.stride(0), a.stride(1), b.stride(0), b.stride(1), c.stride(0), c.stride(1),
                        BM=BM, BN=BN, BK=BK, num_warps=num_warps, num_stages=num_stages)
    return c


def matmul_tuned(a, b):
    M, K = a.shape
    _, N = b.shape
    c = torch.empty((M, N), device=a.device, dtype=torch.float16)
    grid = lambda meta: (triton.cdiv(M, meta["BM"]), triton.cdiv(N, meta["BN"]))
    matmul_auto[grid](a, b, c, M, N, K,
                      a.stride(0), a.stride(1), b.stride(0), b.stride(1), c.stride(0), c.stride(1))
    return c


if RUN_MATMUL:
    S = 4096
    a = torch.randn(S, S, device="cuda", dtype=torch.float16)
    b = torch.randn(S, S, device="cuda", dtype=torch.float16)

    def tflops(ms):
        return 2 * S**3 / ms * 1e-9

    for BM, BN, BK, num_warps in [(32, 32, 32, 4), (64, 64, 32, 4), (128, 64, 32, 4), (128, 128, 32, 8)]:
        label = f"{BM}x{BN}x{BK}"
        try:
            ms = triton.testing.do_bench(lambda: matmul(a, b, BM, BN, BK, num_warps))
            print(f"{label:>12} warps={num_warps}: {tflops(ms):6.1f} TFLOPS")
        except Exception as e:
            print(f"{label:>12} warps={num_warps}: 실패 ({type(e).__name__})")

    ms = triton.testing.do_bench(lambda: torch.matmul(a, b))
    print(f"{'torch.matmul':>12}        : {tflops(ms):6.1f} TFLOPS")

    matmul_auto = triton.autotune(
        configs=[
            triton.Config({"BM": 128, "BN": 128, "BK": 32}, num_warps=8, num_stages=3),
            triton.Config({"BM": 128, "BN": 64, "BK": 32}, num_warps=4, num_stages=4),
            triton.Config({"BM": 64, "BN": 64, "BK": 32}, num_warps=4, num_stages=4),
            triton.Config({"BM": 64, "BN": 64, "BK": 32}, num_warps=4, num_stages=2),
        ],
        key=["M", "N", "K"],
    )(matmul_kernel)
    ms = triton.testing.do_bench(lambda: matmul_tuned(a, b))
    print(f"autotune: {tflops(ms):6.1f} TFLOPS, 고른 설정: {matmul_auto.best_config}")

## 더 해 보기

1. 실습 1의 `n`을 키우면 GB/s가 어떻게 달라질까요?
2. `softmax_kernel`에 온도(temperature) 인자를 넣어 보세요.
3. ReLU, Swish 같은 활성화 함수를 커널로 써 보세요.

## 막힐 때

| 증상 | 해결 |
| --- | --- |
| GPU가 연결되지 않았다는 오류 | 런타임 > 런타임 유형 변경 > T4 GPU를 고르고 모두 실행 |
| GPU를 할당받지 못했다는 안내 | 무료 GPU는 사용량이 많으면 막힐 수 있습니다. 옆 사람과 같이 보거나 다른 Google 계정으로 시도합니다 |
| `name 'results' is not defined` | 위 셀을 건너뛰었습니다. 런타임 > 모두 실행 |
| 2번 셀에 `triton_` 이름이 안 보임 | 일부 환경에서는 프로파일러가 이름을 잡지 못합니다. 커널 수만 봐도 됩니다 |
| 실습 2에서 `정답과 같은가: False` | `num`을 `tl.sum(num, axis=0)`으로 나눴는지 봅니다. 그래도 안 되면 정답 셀을 엽니다 |
| 메모리 부족(`CUDA out of memory`) | 런타임 > 세션 다시 시작 후 모두 실행. `n`을 줄입니다 |